<div style="display: flex; justify-content: flex-start; align-items: center;">
    <a href="https://colab.research.google.com/github/msfasha/307307-BI-Methods-Generative-AI/blob/main/20261/Module%206%20-%20Introduction%20to%20CrewAI/CrewAI_Basics_lab.ipynb" target="_blank">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="height: 25px;">
    </a>
</div>

<div style="background-color:#F5EDF4; border-top:3px solid #A83296; padding:16px; border-radius:6px">
<h1 style="color:#493347; margin:0">CrewAI Basics: the Building Blocks, One at a Time</h1>
<p style="color:#56616D; margin:6px 0 0 0">307307 BI Methods for Generative AI - University of Petra</p>
</div>

This is the **gentle introduction** to CrewAI. Start here, then move on to the longer lab (*Building AI Teams with CrewAI*), which adds data, knowledge, memory and guardrails.

Each section adds **one** idea to the one before it, and every idea comes with a small example you can run and change.

| Part | Building block | In one sentence |
|---|---|---|
| 1 | **Agent** | An LLM with a job description (role, goal, backstory) |
| 2 | **Task** and **Crew** | A piece of work with a clear "done", and the container that runs it |
| 3 | **Several agents** | Agents passing work along, like an assembly line |
| 4 | **Tool** | A Python function the agent may call to get facts or act |
| 5 | **Structured output** | An answer your code can use, not just a paragraph |
| 6 | **Flow** | You decide the order of the steps (`@start`, `@listen`, `@router`) |
| 7 | **Flow + Crew** | A Flow for the skeleton, a Crew for the thinking |

Parts 1 to 5 need a Gemini API key. Part 6 uses **no AI at all**, so you can see exactly how a Flow works.

### The philosophy of CrewAI: why it looks the way it does

Before the code, it helps to understand the **idea** the designers had. Once you see it, every class and every argument in the library becomes predictable. (This is our reading of the design, drawn from how the library is built; it is a way to understand it, not an official statement.)

#### Idea 1: Model an AI system like a small company
In Modules 1 to 5 we built one agent by hand and wrote its loop ourselves. CrewAI starts from a different observation: **we already know how to organise people to do complex work.** We hire specialists, give each a job description, hand out work orders, and let them pass results along. So CrewAI gives you the same vocabulary:

| In a company | In CrewAI |
|---|---|
| An employee with a job title and experience | **Agent** (`role`, `goal`, `backstory`) |
| A work order: what to do, what the deliverable looks like | **Task** (`description`, `expected_output`) |
| A team or department | **Crew** |
| The way work moves (assembly line, or a manager delegating) | **Process** (`sequential` or `hierarchical`) |
| The equipment employees may use | **Tools** |
| The company handbook / the team's notebook | **Knowledge** / **Memory** |
| The standard operating procedure | **Flow** |

This is why the code reads almost like a staffing plan, and why you can explain a crew to a non-programmer.

#### Idea 2: Declare *what* you want, not *how* to loop
When we wrote `TinyAgent` we coded the THOUGHT, ACTION, OBSERVATION loop ourselves. In CrewAI that loop is **built in**. You **declare** who the agent is, what the task is and what a good answer looks like, and the framework handles prompting, calling tools, retrying and passing results to the next agent. The price of this convenience is that the framework builds the prompts for you (we printed one in the Agent section), so you must learn which boxes influence which part of the prompt.

#### Idea 3: Specialists beat generalists
One model asked to "research, analyse and write the report" tends to do everything shallowly. Splitting the job among agents with **narrow, clearly written roles** and a **reviewable output at each step** usually gives better and more checkable results. Each agent sees only what its task needs. The cost is real: more agents means more model calls, more tokens and more waiting (the long lab measures this).

#### Idea 4: Two kinds of control, autonomy *and* determinism
This is the most important idea, and the reason CrewAI has two main building blocks:

| | **Crew** (autonomy) | **Flow** (determinism) |
|---|---|---|
| Philosophy | "Here is the goal and the team; **figure it out**." | "Here is the procedure; **follow it exactly**." |
| Order decided by | the agents (or a manager agent) | **your Python code** |
| Strength | flexible, handles open-ended problems | predictable, cheap, auditable |
| Weakness | can wander, costs more, harder to predict | you must know the steps in advance |

CrewAI's position is that real applications need **both**: a Flow supplies the dependable skeleton (checks, branches, stopping rules, saving results) and Crews supply intelligence inside the steps where judgement is needed. Let the model think where thinking helps; use ordinary code where certainty matters.

#### Idea 5: Make results something other code can trust
A paragraph is fine for a person but useless for a program. So CrewAI puts the **definition of done** into the Task (`expected_output`), lets you demand a **typed object** (`output_pydantic`), lets you add **guardrails** that reject bad answers, and lets a **human approve** (`human_input`). The philosophy: *a model's output is a draft until something checks it.*

#### Idea 6: Batteries included
Tools, knowledge (searchable documents), memory (remembering between runs), delegation, many model providers behind one `LLM(...)` line: you assemble these as options on the objects you already know rather than writing glue code. The aim is a **fast path from idea to working prototype**.

#### Honest trade-offs
A good engineer knows when *not* to use a tool:
- **Cost and speed.** Every agent runs its own thinking loop. A crew is often 5 to 20 times more expensive than a single model call. Use it when the quality gain is worth it.
- **Unpredictability.** Models are not deterministic. The same crew can give different answers on two runs. Flows, guardrails and structured output exist precisely to contain this.
- **Hidden prompts.** Convenience means the framework writes parts of the prompt for you. When results are odd, inspect the prompt and the verbose log.
- **Not always needed.** A single well-written prompt, or one agent with one tool, solves many problems. Add agents only when there is a real reason (different skills, a need for review, steps that must be checked).

**Rule of thumb:** *start with the smallest thing that works (one agent), add a tool when facts matter, add a second agent when a different skill or a reviewer is needed, and wrap it all in a Flow when the process must be predictable.* That is exactly the order of the parts in this material.

#### Questions to think about
1. Which of the six ideas explains why a Task has an `expected_output` box?
2. A bank wants an AI system that approves loans. Which parts would you give to a Crew and which to a Flow, and why?
3. Name a task where a single prompt is better than a crew.


<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Setup</h2>
</div>

Install CrewAI with Gemini support and give it your key. In Google Colab store the key as `GOOGLE_API_KEY` in the Secrets panel (key icon on the left); elsewhere the cell asks you to paste it. The first install takes one to two minutes; if Colab asks you to restart the session, do so and run this cell again.

In [ ]:
# Lines starting with "!" are shell commands. Remove the "!" if you run this in a terminal.
!pip install -q "crewai[google-genai]"

In [ ]:
import os, getpass

# Read the API key from Colab Secrets if available, otherwise ask for it.
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    if not os.environ.get("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")

# Turn off anonymous usage reporting.
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["CREWAI_TRACING_ENABLED"] = "false"

from crewai import Agent, Task, Crew, Process, LLM

# The prefix "gemini/" selects Google Gemini. If you get "model not found", change only this line.
llm = LLM(model="gemini/gemini-3.6-flash")
print("Ready")

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Part 1: The Agent</h2>
</div>

An **agent** is a language model with a *job description*:

- **role**: who it is ("Travel Planner")
- **goal**: what it tries to achieve
- **backstory**: how it behaves and what it values

CrewAI turns these three sentences into the instructions the model receives. Write them as you would write a job advert.

`agent.kickoff("...")` runs the agent once on a question. Nothing else is needed.

In [ ]:
guide = Agent(
    role="Student Travel Guide",
    goal="Give short, practical travel advice to students on a small budget",
    backstory="You have backpacked across 40 countries and you always think about cost first.",
    llm=llm,
)

answer = guide.kickoff("I have 2 days in Amman. What should I see?")
print(answer.raw)

**Try it.** Change only the `backstory` (for example: *"You only recommend luxury hotels"*) and run the cell again. Same model, same question, very different answer. The persona matters.

### Anatomy of an Agent: role, goal and backstory

An LLM on its own is a general-purpose text machine. It has no job, no opinions and no point of view. The three required boxes of an Agent give it one. They are **not decoration**: CrewAI pastes them, almost word for word, into the **system prompt**, the hidden instruction that is sent to the model before your question (Module 2). Everything the model says afterwards is shaped by that text.

For our travel guide, this is **the real text CrewAI sends** (we printed it from the installed library):

```
You are Student Travel Guide. You have backpacked across 40 countries and you always think about cost first.
Your personal goal is: Give short, practical travel advice to students on a small budget
Current Task: <the task description goes here>

Provide your complete response:
```

Match it to the three boxes:

| Box | What it is | How it appears in the prompt | How it changes behaviour |
|---|---|---|---|
| **`role`** | The agent's **job title**: *who it is*. Short, 2 to 4 words. | `You are Student Travel Guide.` | Sets **vocabulary, expertise and viewpoint**. A "Cautious Accountant" and a "Marketing Enthusiast" answer the same question differently. |
| **`goal`** | **What it is trying to achieve**, and what it values. One sentence. | `Your personal goal is: ...` | Sets the **priority** when there is a trade-off. "Short, practical advice for students on a small budget" pushes it towards cheap, brief answers. |
| **`backstory`** | **Experience and character**: *how* it behaves and what it has seen. 1 to 3 sentences. | Placed straight after the role: `You have backpacked across 40 countries...` | Sets **style, habits and attitudes** ("always think about cost first", "never invent facts", "admits uncertainty"). |

(The word is **backstory**, as in the story behind a character in a film: where the character comes from and why it acts the way it does.)

**An important truth.** These three fields do not give the model new knowledge. A "travel guide" agent does not know more about Amman than the same model without the persona. What changes is **focus, tone and priorities**. The persona is a steering wheel, not an engine.

**Why it works.** A language model continues text in the most likely way. After "You are a careful accountant who has seen many businesses fail by launching too fast", the most likely continuation is cautious and numbers-first. After "You are an enthusiastic marketer", it is optimistic and creative. Same model, different continuation.

**Writing good ones**
- **Role**: a real job title, not a sentence. Good: `Market Researcher`. Weak: `Someone who helps with things`.
- **Goal**: one outcome the agent can be judged on. Good: `Find accurate facts and never invent numbers`. Weak: `Be helpful`.
- **Backstory**: behaviour and values, not biography trivia. Good: `You are sceptical and always say when the data is missing`. Weak: `You were born in 1985`.
- **Keep them consistent with each other and with the task.** A `role` of "Poet" with a task of "calculate VAT" will fight itself.
- Do not put the *task* in the backstory. Tasks go in `Task(description=...)`; the agent is a **reusable employee**, the task is **today's assignment**. One agent can do many tasks.

**Try it (no API key needed).** Create two agents with the same `role` but opposite `backstory` sentences and print them with `print(agent.backstory)`. Then, with a key, give both the same question using `.kickoff(...)` and compare.

#### The other Agent boxes you will meet
| Box | Meaning |
|---|---|
| `llm` | **Which model** powers this agent. Different agents may use different models. |
| `tools` | The list of **tools** it may call (Part 4). |
| `verbose=True` | **Print its thinking** step by step. Very useful while learning; switch off in production. |
| `max_iter` | A **safety limit** on how many think-act loops it may take, so it cannot loop forever. |
| `allow_delegation` | May it **ask other agents for help**? Needed for the manager in a hierarchical crew. |
| `max_rpm` | Maximum **requests per minute**, to stay within free-tier limits. |
| `memory`, `knowledge_sources` | Facts it can **remember** between runs, and **documents** it can search (long lab, Steps 5 and 6). |


**What is `kickoff`?** Creating an `Agent` only *describes* it; nothing is sent to the model. `kickoff(...)` is the command that **starts the work** and returns the answer (the name comes from the football *kick-off*, the moment the match begins). `.raw` is the attribute of the returned result that holds the **answer as plain text**. A full explanation of `kickoff` and the related methods follows in Part 2.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Part 2: Task and Crew</h2>
</div>

A chat question is vague. A **Task** is a precise piece of work with two parts:

- `description`: what to do
- `expected_output`: what a finished result looks like (this is your most powerful control)

A **Crew** is the container that holds agents and tasks and runs them with `kickoff()`.

| Component | Answers |
|---|---|
| Agent | *Who* does the work? |
| Task | *What* exactly, and what is "done"? |
| Crew | *Which* agents and tasks run together? |

In [ ]:
explain = Task(
    description="Explain what an API is to a 12-year-old.",
    expected_output="Exactly 3 sentences, including one everyday analogy.",
    agent=guide,
)

crew = Crew(agents=[guide], tasks=[explain])
result = crew.kickoff()
print(result.raw)

**Try it.** Change `expected_output` to *"A single sentence of at most 15 words"* and run again. Notice how much control one line gives you.

### Anatomy of a Task: what goes inside the brackets?

Look again at the code we just wrote:

```python
explain = Task(
    description="Explain what an API is to a 12-year-old.",
    expected_output="Exactly 3 sentences, including one everyday analogy.",
    agent=guide,
)
```

`Task(...)` is a **form with named boxes**. Each line inside the brackets, such as `description="..."`, fills in one box. The box on the left of the `=` is the **field name** (CrewAI decides these names, you cannot invent new ones), and the value on the right is what you put in. So yes: a Task has a fixed structure, and everything you know about the job must fit into these boxes. Think of a work-order form in a company: "what to do", "what we expect", "who does it".

Only **two boxes are compulsory**: `description` and `expected_output`. Everything else is optional and has a sensible default. The table lists the ones you will actually use, most important first.

| Field | Required? | Plain meaning | Example |
|---|---|---|---|
| `description` | **Yes** | **What to do.** Written like an instruction to a colleague. May contain `{placeholders}` that are filled in at `kickoff(inputs=...)`. | `"List 5 facts about {topic}."` |
| `expected_output` | **Yes** | **What "done" looks like.** The agent reads this to know the format, length and level of detail. It is the strongest way to control the result. | `"A bulleted list of 5 facts."` |
| `agent` | Normally yes | **Who** does it. Must be an `Agent` you created earlier. (If you use a manager agent, the manager chooses and you can leave this out.) | `agent=researcher` |
| `context` | No | **Which earlier tasks' results this task needs.** A list of Task objects. | `context=[research]` |
| `tools` | No | Tools for **this task only**, instead of (more precisely: limiting) the agent's usual tools. | `tools=[add_vat]` |
| `output_pydantic` | No | Return the answer as a **typed Python object** with fixed fields instead of free text (Part 5). | `output_pydantic=TripVerdict` |
| `output_json` | No | Like the above, but the result is a plain dictionary. | `output_json=TripVerdict` |
| `output_file` | No | **Save the answer to a file** automatically. | `output_file="report.md"` |
| `guardrail` | No | A Python function that **checks the answer**; if the check fails, the agent must try again (long lab, Step 7). | `guardrail=my_check_function` |
| `human_input` | No | **Pause and ask a person** to approve the answer before continuing. | `human_input=True` |
| `async_execution` | No | Run this task **in the background** while the next one starts. Advanced. | `async_execution=True` |
| `name` | No | A short label that appears in logs. Nice for readability. | `name="research"` |
| `callback` | No | A function CrewAI calls **after** the task finishes (for example, to log or send a message). | `callback=my_function` |

**The three that matter first:** `description` (what), `expected_output` (what good looks like) and `agent` (who). Add `context` as soon as one task depends on another. Reach for the others only when you have a reason.

**Writing a good Task**
- `description` says *what to do*. `expected_output` says *how the answer should look*. Do not mix them: "Explain APIs" is the description; "3 sentences with an analogy" is the expected output.
- Be concrete. "A summary" is weak; "One paragraph of at most 50 words" is strong.
- If you can check the `expected_output` yourself in ten seconds (count sentences, count bullets), the agent can aim at it too.

### The same idea for Agent and Crew

Agent and Crew are also forms with fixed boxes. The important ones:

| Object | Compulsory boxes | Useful optional boxes |
|---|---|---|
| **Agent** | `role`, `goal`, `backstory` | `llm` (which model), `tools` (what it can call), `verbose` (print its thinking), `max_iter` (limit on thinking loops), `allow_delegation` (may it ask colleagues?), `max_rpm` (requests per minute), `memory`, `knowledge_sources` |
| **Crew** | `agents`, `tasks` | `process` (sequential or hierarchical), `verbose`, `memory`, `knowledge_sources`, `manager_llm` or `manager_agent` (for hierarchical), `max_rpm`, `planning` |

CrewAI will complain with a clear message if you leave a compulsory box empty or misspell a field name. That is a feature: the fixed structure catches mistakes early.


### What does `kickoff` mean? (and the other "action" methods)

Until now, everything we wrote only **described** things. `Agent(...)`, `Task(...)` and `Crew(...)` build objects in memory; they are like writing a job advert, a work order and an org chart. **Nothing happens, no model is called, nothing costs money**, until you tell CrewAI to start. That start command is **`kickoff()`**.

The word comes from football: the *kick-off* is the moment the match begins. So:

> `something.kickoff(...)` means **"stop describing, begin the work now"**. The call waits until the work is finished and **returns the result**.

A **method** is just a function that belongs to an object, written `object.method(...)`. So `crew.kickoff()` means "ask this crew to start". The table below lists every such method you will meet, from the one you use constantly to the ones you can ignore for now.

| Method | Belongs to | What it does | Typical call |
|---|---|---|---|
| **`kickoff()`** | **Crew** | Run **all the tasks** of the crew in order, and return the final result. The values for `{placeholders}` go in `inputs`. | `crew.kickoff(inputs={"topic": "solar"})` |
| **`kickoff()`** | **Agent** | Run **one agent** on **one question** with no Task or Crew. The quickest way to try an agent. The question is passed as text. | `guide.kickoff("Two days in Amman?")` |
| **`kickoff()`** | **Flow** | Run the **whole flow**: begin at the `@start` step and follow the `@listen` and `@router` steps. | `OrderFlow().kickoff()` |
| `kickoff_for_each()` | Crew | Run the **same crew many times**, once per item of a list of inputs, and return a list of results. | `crew.kickoff_for_each([{"topic": "solar"}, {"topic": "wind"}])` |
| `kickoff_async()` / `akickoff()` | Crew, Agent | Start the work **without blocking** your program, so you can do something else meanwhile (in notebooks use `await`). Advanced. | `await crew.kickoff_async(inputs=...)` |
| `test()` | Crew | Run the crew several times and have a judge model **score the quality**. Used by developers when tuning a crew. | `crew.test(n_iterations=2, eval_llm="gemini/...")` |
| `train()` | Crew | Run the crew while a person **gives feedback**, which is saved so agents improve. Advanced. | `crew.train(n_iterations=2, filename="t.pkl")` |
| `replay()` | Crew | **Re-run from a chosen task** of an earlier run, without repeating the earlier ones. Advanced. | `crew.replay(task_id="...")` |
| `run()` | **Tool** | Run a tool **directly** with no AI, to test it. | `add_vat.run(price=80, vat_percent=16)` |
| `call()` | **LLM** | Send **one message straight to the model**, with no agent around it. | `llm.call("Say hello")` |

In this course you will use **`kickoff()`** (on Crew, Agent and Flow) all the time, `tool.run()` to test tools, and perhaps `kickoff_for_each()`. The rest are there so you recognise them in the documentation.

#### `inputs`: filling the placeholders
`inputs` is a dictionary (`{"name": value}`). Every `{name}` found inside a task `description`, an `expected_output`, or an agent's `role`, `goal` or `backstory` is replaced by the value before the work starts. That is why one crew can serve many topics:

```python
crew.kickoff(inputs={"topic": "solar power"})     # {topic} becomes "solar power"
```

#### What does `kickoff()` give back?
It returns a **result object**, not just text. Think of a parcel: the answer is inside, with a label on it. The most useful attributes (read them with a dot):

| Attribute | What it holds |
|---|---|
| `result.raw` | **The final answer as plain text.** This is what you print most of the time. |
| `result.pydantic` | The answer as a typed object, **if** the last task used `output_pydantic` (Part 5). |
| `result.json_dict` | The answer as a dictionary, **if** the last task used `output_json`. |
| `result.tasks_output` | A **list with one entry per task**, so you can see what each step produced. `result.tasks_output[0].raw` is the first task's text. |
| `result.token_usage` | **How much text the run consumed.** Tokens are what you pay for (Module 2). |

(`Agent.kickoff()` returns a similar object; `.raw` works there too.)


In [ ]:
# kickoff returns a result object. Look inside it (this reuses `result` from the cell above).
print("Type of result:", type(result).__name__)
print("--- result.raw (the final answer as text) ---")
print(result.raw)
print("--- result.tasks_output (one entry per task) ---")
for t in result.tasks_output:
    print("*", t.description, "->", t.raw[:80], "...")
print("--- result.token_usage (how much text was consumed) ---")
print(result.token_usage)

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Part 3: Several agents passing work along</h2>
</div>

Real work often needs different skills. Here a **researcher** collects facts and a **writer** turns them into a short summary. By default a crew runs tasks **one after another** (`Process.sequential`), like an assembly line, and each task receives the earlier results.

Two small features make the crew reusable:

- `{topic}` is a **placeholder** in a task description. Its value is supplied at `kickoff(inputs={...})`.
- `context=[research]` says explicitly: *this task needs the output of that task*.

In [ ]:
researcher = Agent(
    role="Researcher",
    goal="List accurate, relevant facts",
    backstory="Thorough and sceptical. You never invent facts.",
    llm=llm,
)
writer = Agent(
    role="Writer",
    goal="Write crisp summaries for busy managers",
    backstory="You write short, clear business prose.",
    llm=llm,
)

research = Task(
    description="List 5 facts about {topic}.",
    expected_output="A bulleted list of 5 facts.",
    agent=researcher,
)
write = Task(
    description="Turn the facts into a summary for a manager.",
    expected_output="One paragraph of at most 50 words.",
    agent=writer,
    context=[research],        # the writer receives the researcher's output
)

team = Crew(agents=[researcher, writer], tasks=[research, write], process=Process.sequential)

print(team.kickoff(inputs={"topic": "solar power in Jordan"}).raw)

**Try it.** Run the *same* `team` again with `inputs={"topic": "electric cars"}`. One crew, any topic. Then make `research.expected_output` vague (*"Some facts"*) and see how the quality drops.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Part 4: Tools</h2>
</div>

A language model is fluent but **bad at exact arithmetic and live facts**. A **tool** is an ordinary Python function the agent may choose to call.

The agent never sees your code. It sees the tool's **name**, its **type hints** and its **docstring** (the text in triple quotes). That text is how it decides *when* and *how* to use the tool, so write it carefully.

Remember *intent vs action* from Module 4: the model only *asks* for the tool; CrewAI runs it and gives the result back.

In [ ]:
from crewai.tools import tool

@tool("Add VAT")
def add_vat(price: float, vat_percent: float) -> str:
    """Calculate the final price after adding VAT (sales tax).
    Inputs: the price before tax, and the VAT rate as a percentage (for example 16 for 16%).
    Returns the price including VAT, rounded to 2 decimals."""
    return str(round(price * (1 + vat_percent / 100), 2))

# You can test a tool directly, with no AI involved:
print(add_vat.run(price=80, vat_percent=16))

### Why the docstring matters: how an agent "reads" a tool

In ordinary Python, a **docstring** (the text in triple quotes right under the `def` line) is only a comment for humans. Python ignores it when running the function. **In CrewAI it is not a comment: it is the instruction manual the model reads.** Here is why.

**The model never sees your code.** An LLM only deals with text. It cannot open your function, read `price * (1 + vat_percent / 100)` and understand it. When an agent is created with `tools=[add_vat]`, CrewAI extracts **three things** from your function and puts them in the prompt:

| What CrewAI extracts | Where it comes from | What the model does with it |
|---|---|---|
| **Name** | `@tool("Add VAT")` | Uses it to refer to the tool when it decides to call it. |
| **Description** | The **whole docstring** | Decides **whether** this tool helps with the task, and **what to pass in**. |
| **Arguments and their types** | The **type hints**: `price: float, vat_percent: float` | Knows which values it must supply and their kind (number, text, ...). |

We printed what CrewAI really stores for our tool:

```
NAME: Add VAT
DESCRIPTION: Calculate the final price after adding VAT (sales tax).
    Inputs: the price before tax, and the VAT rate as a percentage (for example 16 for 16%).
    Returns the price including VAT, rounded to 2 decimals.
ARGUMENTS: price (number, required), vat_percent (number, required)
```

That block of text is **everything the model knows** about the tool. If it is vague, the model guesses.

**The flow, step by step**
1. You ask: "A laptop costs 849.50 before tax, VAT is 16%. What is the final price?"
2. The model reads its tool list. It sees *Add VAT: Calculate the final price after adding VAT...* and judges: "This matches, and I have both inputs."
3. It writes a **tool call**: `Add VAT(price=849.50, vat_percent=16)`. (Intent only; Module 4.)
4. **CrewAI runs your real Python function** and returns `985.42` as an Observation.
5. The model writes the final answer using that number.

Step 2 is decided **entirely by the docstring**. Step 3 is shaped by the docstring *and* the type hints.

**Three rules that follow**
1. **A docstring is compulsory.** If you write `@tool` on a function with no docstring, CrewAI stops with the error `Function must have a docstring`. It refuses to give the model a tool with no manual.
2. **Type hints are compulsory in practice.** Without `: float` or `: str` the model does not know what kind of value to send.
3. **Write the docstring for the model, not for yourself.** State (a) **what the tool does**, (b) **what each input means, with units and an example** ("VAT rate as a percentage, e.g. 16 for 16%"), (c) **what it returns**, and (d) **when to use it** or any limits ("the product must be one of: Espresso, Latte...").

**Why unit and example matter.** Our docstring says *"for example 16 for 16%"*. Without it the model might send `0.16` and the tool would add almost nothing. The function would still run without any error and quietly return a wrong number. Many "the agent is dumb" bugs are really "the docstring was ambiguous".

**Good versus weak**

| Weak (the agent guesses) | Strong (the agent knows) |
|---|---|
| `"""Does tax stuff."""` | `"""Calculate the final price after adding VAT (sales tax). Inputs: price before tax, and the VAT rate as a percentage (for example 16 for 16%). Returns the price including VAT, rounded to 2 decimals."""` |
| `"""Gets data."""` | `"""Summarise twelve months of sales for ONE product. The product must be one of: Espresso, Latte, Hot Tea. Returns total units and revenue."""` |

**Other good habits**: one tool = one job (narrow tools are used more reliably than one tool with ten options); a clear, specific name (`Add VAT`, not `Tool1`); return **short, readable text** (the model has to read the result); and if the input is invalid, return a **helpful message** (`"Unknown product. Use one of: ..."`) instead of crashing, so the agent can correct itself and try again.

**Python connection.** This is why in Python the docstring is also the text shown by `help(add_vat)`. CrewAI reuses a convention you already know and gives it a new audience: **the AI**.


In [ ]:
# See exactly what the agent will be told about the tool (no AI involved).
print("NAME:       ", add_vat.name)
print("DESCRIPTION:", add_vat.description)
print("ARGUMENTS:  ", list(add_vat.args_schema.model_json_schema()["properties"]))

Now give the tool to an agent. The only change is `tools=[...]`. Watch the printed log: you will see the agent decide to call `add_vat`, the numbers it sent, and the result it received.

In [ ]:
shopper = Agent(
    role="Shopping Assistant",
    goal="Answer price questions exactly, never guessing a number you can calculate",
    backstory="You help customers understand what they will really pay.",
    tools=[add_vat],
    llm=llm,
    verbose=True,
)

price_task = Task(
    description="A laptop costs 849.50 before tax. VAT is 16%. What is the final price?",
    expected_output="One sentence stating the final price.",
    agent=shopper,
)

print(Crew(agents=[shopper], tasks=[price_task]).kickoff().raw)

**Try it.** Make the docstring vague (delete the "Inputs:" lines) and ask again. Does the agent still use the tool correctly? A weak docstring is the most common reason an agent misuses a tool.

**Exercise.** Write a tool `tip_calculator(bill, tip_percent)` that returns the tip amount, test it with `.run(...)`, and give it to an agent.

In [ ]:
# YOUR CODE HERE



# ---- Solution (remove the # signs to try it) ----
# @tool("Tip calculator")
# def tip_calculator(bill: float, tip_percent: float) -> str:
#     """Calculate the tip for a restaurant bill.
#     Inputs: the bill amount and the tip as a percentage (for example 10 for 10%).
#     Returns the tip amount rounded to 2 decimals."""
#     return str(round(bill * tip_percent / 100, 2))
#
# print(tip_calculator.run(bill=45, tip_percent=10))   # expect 4.5

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Part 5: Structured output</h2>
</div>

Free text is hard to use in a program. If a later step must read a *field* (a score, a yes/no, a list), ask for a typed object. We describe its shape with a small Python class (a **Pydantic model**) and give it to the task as `output_pydantic`.

In [ ]:
from typing import List, Literal
from pydantic import BaseModel

class TripVerdict(BaseModel):
    worth_it: Literal["yes", "maybe", "no"]    # only these three values are allowed
    reasons: List[str]

verdict_task = Task(
    description="A student asks: is a 1-day trip from Amman to Petra worth the cost?",
    expected_output="A verdict (yes, maybe or no) and 2 to 3 short reasons.",
    agent=guide,
    output_pydantic=TripVerdict,
)

out = Crew(agents=[guide], tasks=[verdict_task]).kickoff()

print(out.pydantic.worth_it)       # a real Python value, usable in an if-statement
print(out.pydantic.reasons)

**Check.** `out.pydantic.worth_it` is always one of three exact words. That is what lets the next part make a decision with a plain `if`.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Part 6: Flows (no AI needed)</h2>
</div>

A crew lets agents *decide*. Sometimes you want **you** to decide the order: do this, then that, and *if* something then do something else. That is a **Flow**.

A Flow is a normal Python class where **each method is a step**. Decorators say *when* a step runs:

| Decorator | Meaning |
|---|---|
| `@start()` | Run this when the flow begins (the entry point) |
| `@listen(step)` | Run this after `step` finishes; it receives `step`'s return value |
| `@router(step)` | Run after `step`, then **choose a path** by returning a text label |
| `@listen("label")` | Run when a router returns that label |

Think of **dominoes**: `@start` tips the first one and `@listen` says which falls next. `self.state` is memory shared by every step.

First a Flow with **no AI at all**, so you only see the mechanics. This one needs no API key.

In [ ]:
from crewai.flow.flow import Flow, start, listen, router
from pydantic import BaseModel

class OrderState(BaseModel):
    amount: float = 0

class OrderFlow(Flow[OrderState]):

    @start()
    def receive_order(self):
        print("1. order received")
        self.state.amount = 250          # shared memory for the whole flow
        return "order-17"

    @listen(receive_order)
    def log_order(self, order_id):       # gets receive_order's return value
        print("2. logging", order_id, "amount =", self.state.amount)

    @router(receive_order)
    def check_amount(self):
        return "needs_approval" if self.state.amount > 100 else "auto_approve"

    @listen("needs_approval")
    def ask_manager(self):
        print("3. amount is large, a manager must approve")

    @listen("auto_approve")
    def approve(self):
        print("3. small amount, approved automatically")

OrderFlow().kickoff()

Read the output and match it to the code: `receive_order` ran first (`@start`); `log_order` and the router both ran after it; the router returned `"needs_approval"`, so **only** `ask_manager` ran, never `approve`. That is an if/else.

**Questions.**
1. Change `self.state.amount = 250` to `50` and predict the output before you run it. Does `log_order` still run?
2. Why is the router better than an `if` buried inside one big function? (Hint: each step is small, named and easy to read.)

In [ ]:
# YOUR CODE HERE: change the amount, run OrderFlow().kickoff() again

A flow can also **wait for several steps**. With `and_(a, b)` a step runs only after *both* finish; with `or_(a, b)` as soon as *either* finishes.

In [ ]:
from crewai.flow.flow import and_

class CheckFlow(Flow):
    @start()
    def check_stock(self):
        print("stock checked")

    @start()
    def check_payment(self):
        print("payment checked")

    @listen(and_(check_stock, check_payment))
    def ship(self):
        print("both checks done: shipping")

CheckFlow().kickoff()

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Part 7: Flow + Crew together</h2>
</div>

The usual real-world pattern: **a Flow for the skeleton, Crews for the thinking.** Each step of a Flow can be plain Python, one agent, or a whole crew.

Here we do a cheap one-agent check first (Part 5's verdict). A router then **stops early** on a "no" and only spends on the full two-agent team (Part 3) when the answer is promising. This saves money and makes the process predictable.

In [ ]:
class TripState(BaseModel):
    question: str = ""
    worth_it: str = ""
    plan: str = ""

class TripFlow(Flow[TripState]):

    @start()
    def quick_check(self):
        t = Task(
            description="Quick opinion for a student: {question}",
            expected_output="A verdict (yes, maybe or no) and 2 to 3 short reasons.",
            agent=guide,
            output_pydantic=TripVerdict,
        )
        out = Crew(agents=[guide], tasks=[t]).kickoff(inputs={"question": self.state.question})
        self.state.worth_it = out.pydantic.worth_it

    @router(quick_check)
    def decide(self):
        return "stop" if self.state.worth_it == "no" else "go"

    @listen("stop")
    def say_no(self):
        print("Quick check says no. Stopping here, the full team was never used.")

    @listen("go")
    def full_plan(self):
        r = Task(description="List 5 facts a student should know before this trip: {question}",
                 expected_output="A bulleted list of 5 facts.", agent=researcher)
        w = Task(description="Write a short plan from the facts.",
                 expected_output="One paragraph of at most 60 words.", agent=writer, context=[r])
        out = Crew(agents=[researcher, writer], tasks=[r, w]).kickoff(inputs={"question": self.state.question})
        self.state.plan = out.raw
        print(self.state.plan)

flow = TripFlow()
flow.state.question = "a weekend trip from Amman to Aqaba"
flow.kickoff()

**Try it.** Run the flow with a question that should get a "no" (for example *"a 1-day trip from Amman to Tokyo"*) and watch it stop early.

## Crew or Flow?

| | **Crew** | **Flow** |
|---|---|---|
| Who decides the order? | The agents | You, in code |
| Strength | Open-ended, flexible thinking | Predictable, cheap, easy to audit |
| Use for | "Research this and write a summary" | "Check first; only then spend on the full team" |

In practice you combine them, exactly as in Part 7.

<div style="background-color:#F7F3F7; border-left:4px solid #C8ABC3; padding:10px; border-radius:4px; margin-top:14px">
<h2 style="color:#493347; margin:0">Summary</h2>
</div>

- An **Agent** is a model with a role, goal and backstory. A **Task** says what to do and what "done" looks like (`expected_output`). A **Crew** runs them.
- With several agents, tasks run in order and pass results along; `{placeholders}` and `inputs` make a crew reusable.
- A **Tool** is a Python function; its **docstring** is how the agent understands it.
- **Structured output** (`output_pydantic`) gives code a field it can use.
- A **Flow** gives you control: `@start` begins, `@listen` follows, `@router` branches, and `and_` / `or_` wait for several steps.
- Use **Crews to think and Flows to control.**

**Next:** open the longer lab, *Building AI Teams with CrewAI*, which applies these blocks to a small business with real data, then adds knowledge, memory, guardrails and a manager agent.